# Bike Sharing Usage & Operations Analysis

## 1. Data Loading & Initial Exploration

This notebook explores Divvy bike-sharing trip data for January–July 2026.
The goal of this stage is to inspect the source files, combine the monthly datasets, and identify data-quality issues before cleaning.

In [2]:
import pandas as pd
from pathlib import Path

# Folder containing the CSV files
data_folder = Path.cwd()

# List for storing monthly DataFrames
data_parts = []

for month in range(1, 8):
    file_name = f"{month:02d}.csv"
    file_path = data_folder / file_name

    df_month = pd.read_csv(file_path)

    print(f"{file_name}: {df_month.shape[0]} rows")

    data_parts.append(df_month)

# Combine all months into one DataFrame
df = pd.concat(data_parts, ignore_index=True)

print("\nCombined dataset:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

01.csv: 137787 rows
02.csv: 201450 rows
03.csv: 317037 rows
04.csv: 448252 rows
05.csv: 653704 rows
06.csv: 762550 rows
07.csv: 869051 rows

Combined dataset:
(3389831, 13)

Columns:
['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']

First 5 rows:


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,FD0EB1D32AF0D47E,classic_bike,2026-01-31 09:13:09.018,2026-01-31 09:28:10.302,Central St & Girard Ave,CHI02042,Dodge Ave & Church St,CHI00741,42.064313,-87.686152,42.048308,-87.698224,member
1,FB27405C3F8C824F,classic_bike,2026-01-15 14:25:42.526,2026-01-15 14:33:18.854,Shore Dr & 55th St,CHI00394,Woodlawn Ave & 55th St,CHI00423,41.795212,-87.580715,41.795264,-87.596471,casual
2,6FAFA1709403AA27,electric_bike,2026-01-06 12:55:33.572,2026-01-06 13:02:17.922,Hampden Ct & Diversey Pkwy,CHI02087,NaN,NaN,41.932470,-87.642420,41.940000,-87.640000,member
3,1F34C1FAD9FEC2D8,electric_bike,2026-01-26 16:22:25.011,2026-01-26 16:53:15.197,Carpenter St & Huron St,CHI00286,NaN,NaN,41.894532,-87.653412,41.830000,-87.670000,member
4,8E3E3072D8D3D918,electric_bike,2026-01-10 18:13:30.139,2026-01-10 19:31:56.971,Clinton St & Madison St,CHI00233,NaN,NaN,41.881660,-87.641150,41.890000,-87.630000,member


## 2. Data Quality Check

Before cleaning the dataset, I checked data types, missing values, duplicates, date coverage, and categorical values.

In [1]:
print("Dataset shape:")
print(df.shape)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
missing = df.isna().sum()
missing_percent = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_percent
})

display(missing_summary)

print("\nDuplicate ride IDs:")
print(df["ride_id"].duplicated().sum())

print("\nRideable types:")
print(df["rideable_type"].value_counts())

print("\nUser types:")
print(df["member_casual"].value_counts())

print("\nRaw date range:")
print(df["started_at"].min())
print(df["started_at"].max())

Dataset shape:


NameError: name 'df' is not defined

### Initial Data Quality Findings

The combined dataset contains 3,389,831 trip records.
Station information is missing for approximately 20–21% of trips, while coordinate data is almost complete. Only 35 duplicate ride IDs were detected.
The raw date range starts on 31 December 2025, although the analyzed source files cover January–July 2026. These records will be investigated before cleaning.
Datetime columns are currently stored as strings and require conversion before time-based analysis.

In [ ]:
# Convert date columns to datetime
df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"] = pd.to_datetime(df["ended_at"])

# Check records outside the expected analysis period
outside_period = df[
    (df["started_at"] < "2026-01-01") |
    (df["started_at"] >= "2026-08-01")
]

print("Records outside Jan-Jul 2026:")
print(len(outside_period))

display(outside_period.head())

# Inspect duplicate ride IDs
duplicate_ids = df[df["ride_id"].duplicated(keep=False)].sort_values("ride_id")

print("\nRows with duplicate ride IDs:")
print(len(duplicate_ids))

display(duplicate_ids.head(10))

Records outside Jan-Jul 2026:
23


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
246,427AB6A371D47BDC,electric_bike,2025-12-31 23:48:36.449,2026-01-01 00:13:32.821,Carpenter St & Huron St,CHI00286,Carpenter St & Huron St,CHI00286,41.894556,-87.653449,41.894556,-87.653449,casual
1610,024231BAC90568CA,classic_bike,2025-12-31 23:36:22.998,2026-01-01 00:39:10.110,DuSable Lake Shore Dr & North Blvd,CHI00525,DuSable Lake Shore Dr & North Blvd,CHI00525,41.911722,-87.626804,41.911722,-87.626804,casual
12038,BB94A18BA0462AC5,classic_bike,2025-12-31 23:31:12.103,2026-01-01 02:26:23.361,Canal St & Madison St,CHI00498,Sedgwick St & Schiller St,CHI00303,41.882409,-87.639767,41.907626,-87.638566,casual
12061,7EF8E81EDA2F4431,classic_bike,2025-12-31 23:50:13.442,2026-01-01 00:15:45.922,Indiana Ave & 31st St,CHI00436,Shedd Aquarium,CHI00533,41.838842,-87.621857,41.867226,-87.615355,member
13480,FCDF0EB49F099E61,electric_bike,2025-12-31 23:14:24.294,2026-01-01 00:12:07.962,Clark St & Newport St,CHI00674,NaN,NaN,41.944540,-87.654678,41.900000,-87.620000,casual



Rows with duplicate ride IDs:
70


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
900387,03C5375398BE0649,electric_bike,2026-04-30 23:46:36.039,2026-05-01 00:04:48.512,NaN,NaN,Clark St & Arlington Pl,CHI02109,41.900000,-87.630000,41.926940,-87.641480,member
1641241,03C5375398BE0649,electric_bike,2026-04-30 23:46:36.039,2026-05-01 00:04:48.512,NaN,NaN,Clark St & Arlington Pl,CHI02109,41.900000,-87.630000,41.926940,-87.641480,member
1021719,0612B2A21BA5E988,classic_bike,2026-04-30 15:56:11.305,2026-05-01 16:56:08.148,Fulton Market,CHI01880,NaN,NaN,41.886871,-87.648089,NaN,NaN,member
1364300,0612B2A21BA5E988,classic_bike,2026-04-30 15:56:11.305,2026-05-01 16:56:08.148,Fulton Market,CHI01880,NaN,NaN,41.886871,-87.648089,NaN,NaN,member
1256308,10117825A92C182D,classic_bike,2026-04-30 23:56:24.321,2026-05-01 00:01:51.860,Peoria St & Kinzie St,CHI02098,Ogden Ave & Chicago Ave,CHI00275,41.888960,-87.649730,41.896362,-87.654061,casual
840946,10117825A92C182D,classic_bike,2026-04-30 23:56:24.321,2026-05-01 00:01:51.860,Peoria St & Kinzie St,CHI02098,Ogden Ave & Chicago Ave,CHI00275,41.888960,-87.649730,41.896362,-87.654061,casual
773136,15521787039F37E5,classic_bike,2026-04-30 23:54:44.934,2026-05-01 00:05:45.575,Michigan Ave & 11th St,CHI02025,DuSable Lake Shore Dr & Monroe St,CHI00374,41.869590,-87.623961,41.880958,-87.616743,member
1522550,15521787039F37E5,classic_bike,2026-04-30 23:54:44.934,2026-05-01 00:05:45.575,Michigan Ave & 11th St,CHI02025,DuSable Lake Shore Dr & Monroe St,CHI00374,41.869590,-87.623961,41.880958,-87.616743,member
1032059,1DB04E901469FB2F,electric_bike,2026-04-30 23:54:09.726,2026-05-01 00:11:09.709,Ravenswood Ave & Irving Park Rd,CHI00309,NaN,NaN,41.954690,-87.673930,41.970000,-87.670000,casual
1720990,1DB04E901469FB2F,electric_bike,2026-04-30 23:54:09.726,2026-05-01 00:11:09.709,Ravenswood Ave & Irving Park Rd,CHI00309,NaN,NaN,41.954690,-87.673930,41.970000,-87.670000,casual


In [ ]:
# Check whether duplicate ride IDs contain conflicting data
duplicate_rows = df[df["ride_id"].duplicated(keep=False)]

duplicate_check = (
    duplicate_rows
    .groupby("ride_id")
    .nunique(dropna=False)
)

conflicting_duplicates = duplicate_check[
    (duplicate_check > 1).any(axis=1)
]

print("Duplicate ride IDs:", duplicate_rows["ride_id"].nunique())
print("Duplicate IDs with conflicting data:", len(conflicting_duplicates))

Duplicate ride IDs: 35
Duplicate IDs with conflicting data: 0


### Duplicate Validation

35 duplicate ride IDs were identified in the combined dataset. All duplicate records contained identical information, with no conflicting values across columns. These records can therefore be safely deduplicated during the cleaning stage by keeping a single record for each ride ID.